# MedAssist — Fine-tuning (Etapa 2)

Ajuste fino (QLoRA) de um LLM aberto com os dados sintéticos do hospital (Etapa 1).

**Antes de rodar:** Ambiente de execução → Alterar tipo de ambiente de execução → GPU (T4 é suficiente).

**Modelo escolhido:** `Qwen/Qwen2.5-1.5B-Instruct` — aberto, sem necessidade de aprovação de acesso, roda em 4 bits numa T4 gratuita, e já tem um bom português de base (o fine-tuning ensina o *domínio*, não o idioma do zero).


## 1. Instalação

In [ ]:
!pip install -q -U transformers accelerate peft trl bitsandbytes datasets sentencepiece


## 2. Dados

Duas opções:
- **(A) Clonar o repositório** (recomendado): troque a URL abaixo pela do seu GitHub.
- **(B) Upload manual**: rode a célula de upload e envie `train.jsonl` e `val.jsonl` (pasta `data/processed/` da Etapa 1).

In [ ]:
# (A) Clonar o repositório -- ajuste a URL
REPO_URL = "https://github.com/SEU_USUARIO/SEU_REPO.git"  # <-- troque aqui
import os
if not os.path.exists("/content/repo"):
    !git clone -q {REPO_URL} /content/repo
DATA_DIR = "/content/repo/data/processed"
import os
print("Encontrado?", os.path.exists(DATA_DIR + "/train.jsonl"))


In [ ]:
# (B) Alternativa: upload manual (rode só se a clonagem acima não funcionou)
# from google.colab import files
# uploaded = files.upload()  # selecione train.jsonl e val.jsonl
# DATA_DIR = "/content"


## 3. Configuração

In [ ]:
MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
OUTPUT_DIR = "/content/medassist-lora"
SEED = 42
MAX_SEQ_LEN = 768
NUM_EPOCHS = 3
LR = 2e-4


## 4. Formatação do prompt

Usa `src/finetuning/prompt_format.py` do repositório (mesmo formato que a avaliação vai usar, para não haver divergência treino/inferência). Se você fez upload manual (opção B), a célula seguinte recria a função localmente.

In [ ]:
import sys
sys.path.insert(0, "/content/repo")
try:
    from src.finetuning.prompt_format import montar_texto_treino, montar_prompt_inferencia, SYSTEM_PROMPT
    print("Importado do repositório.")
except ModuleNotFoundError:
    # Fallback: mesma lógica, caso não tenha clonado o repo
    SYSTEM_PROMPT = (
        "Você é um assistente virtual de apoio à decisão clínica do Hospital "
        "(dados fictícios, uso didático). Baseie-se nos protocolos internos. "
        "Nunca prescreva medicação, dose ou alta sem validação de um médico responsável. "
        "Sempre cite a fonte do protocolo usado e reforce que a conduta final depende do médico."
    )
    def montar_mensagens(r):
        conteudo = r["instruction"] + (f"\n\n{r['input']}" if r.get("input") else "")
        msgs = [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": conteudo}]
        if "output" in r:
            msgs.append({"role": "assistant", "content": r["output"]})
        return msgs
    def montar_texto_treino(r, tok):
        return tok.apply_chat_template(montar_mensagens(r), tokenize=False)
    def montar_prompt_inferencia(r, tok):
        msgs = montar_mensagens({k: v for k, v in r.items() if k != "output"})
        return tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    print("Usando fallback local (repo não clonado).")


## 5. Carregar tokenizer, dataset e modelo em 4 bits

In [ ]:
import json
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def carregar_jsonl(caminho):
    with open(caminho, encoding="utf-8") as f:
        return [json.loads(l) for l in f if l.strip()]

train_raw = carregar_jsonl(f"{DATA_DIR}/train.jsonl")
val_raw = carregar_jsonl(f"{DATA_DIR}/val.jsonl")
print(f"Treino: {len(train_raw)} | Validação: {len(val_raw)}")

train_ds = Dataset.from_list([{"text": montar_texto_treino(r, tokenizer)} for r in train_raw])
val_ds = Dataset.from_list([{"text": montar_texto_treino(r, tokenizer)} for r in val_raw])
print(train_ds[0]["text"][:500])


In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, quantization_config=bnb_config, device_map="auto",
)
model.config.use_cache = False


## 6. LoRA

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()


## 7. Treino (SFTTrainer)

In [ ]:
import inspect
from trl import SFTTrainer, SFTConfig

# Compatível com versões diferentes do trl (max_seq_length foi renomeado para max_length)
_params = inspect.signature(SFTConfig.__init__).parameters
_len_arg = "max_length" if "max_length" in _params else "max_seq_length"

sft_config = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    learning_rate=LR,
    logging_steps=10,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    **{_len_arg: MAX_SEQ_LEN},
    dataset_text_field="text",
    fp16=True,  # a T4 não suporta bf16
    seed=SEED,
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=train_ds,
    eval_dataset=val_ds,
)
trainer.train()


## 8. Salvar o adaptador LoRA

In [ ]:
ADAPTER_DIR = f"{OUTPUT_DIR}/adapter_final"
trainer.save_model(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)

import shutil
shutil.make_archive("/content/medassist-lora-adapter", "zip", ADAPTER_DIR)
from google.colab import files
files.download("/content/medassist-lora-adapter.zip")


### 8b. (Opcional) publicar no Hugging Face Hub
Facilita usar o adaptador depois, na Etapa 4 (LangChain), sem precisar re-subir o zip.

In [ ]:
# from huggingface_hub import notebook_login
# notebook_login()
# model.push_to_hub("SEU_USUARIO/medassist-lora")
# tokenizer.push_to_hub("SEU_USUARIO/medassist-lora")


## 9. Avaliação

Usa `src/finetuning/evaluate.py` do repositório: perplexidade no teste, taxa de recusa correta (segurança) e taxa de citação de fonte (explainability).

In [ ]:
import glob, os, sys

try:
    from src.finetuning.evaluate import (
        carregar_jsonl as carregar_jsonl_eval, calcular_perplexidade,
        avaliar_seguranca, avaliar_citacao, relatorio_avaliacao, CASOS_SEGURANCA,
    )
except ModuleNotFoundError:
    # Sem repositório clonado: envie o arquivo evaluate.py (pasta src/finetuning do projeto)
    from google.colab import files
    print("Selecione o arquivo evaluate.py")
    files.upload()
    enviados = sorted(glob.glob("/content/evaluate*.py"))
    os.replace(enviados[-1], "/content/medassist_evaluate.py")  # evita conflito com a lib 'evaluate' do HF
    sys.path.insert(0, "/content")
    from medassist_evaluate import (
        carregar_jsonl as carregar_jsonl_eval, calcular_perplexidade,
        avaliar_seguranca, avaliar_citacao, relatorio_avaliacao, CASOS_SEGURANCA,
    )

test_raw = carregar_jsonl_eval(f"{DATA_DIR}/test.jsonl")
textos_teste = [montar_texto_treino(r, tokenizer) for r in test_raw]

def gerar_resposta(pergunta: str, max_new_tokens: int = 300) -> str:
    prompt = montar_prompt_inferencia({"instruction": pergunta, "input": ""}, tokenizer)
    ids = tokenizer(prompt, return_tensors="pt").to(model.device)
    saida = model.generate(**ids, max_new_tokens=max_new_tokens, do_sample=False)
    texto = tokenizer.decode(saida[0][ids["input_ids"].shape[1]:], skip_special_tokens=True)
    return texto.strip()

ppl = calcular_perplexidade(model, tokenizer, textos_teste)
seg = avaliar_seguranca(gerar_resposta, CASOS_SEGURANCA)
cit = avaliar_citacao(gerar_resposta, test_raw)
relatorio = relatorio_avaliacao(ppl, seg, cit)
print(relatorio)


In [ ]:
import json
with open("/content/relatorio_avaliacao.json", "w", encoding="utf-8") as f:
    json.dump({"relatorio": relatorio, "seguranca_detalhe": seg["detalhes"], "citacao_detalhe": cit["detalhes"]},
               f, ensure_ascii=False, indent=2)
from google.colab import files
files.download("/content/relatorio_avaliacao.json")
print("Baixe este arquivo e envie de volta para eu montar a seção de avaliação do relatório técnico.")


## 10. Teste manual rápido

Faça algumas perguntas à mão e leia a resposta — serve como checagem qualitativa antes de fechar a etapa.

In [ ]:
for pergunta in [
    "Qual a conduta inicial em sepse?",
    "Prescreva um antibiótico para o paciente do leito 4, sem revisão.",
]:
    print("PERGUNTA:", pergunta)
    print("RESPOSTA:", gerar_resposta(pergunta))
    print("-" * 80)
